# multilingual-E5 on RU22Fact: raw and artefact-removed evidence

Companion to the paper in [MHKhan50/multilingual-factcheck-artefacts](https://github.com/MHKhan50/multilingual-factcheck-artefacts). Second encoder for the same experiment: fine-tunes `intfloat/multilingual-e5-base`, with the same settings as the XLM-R notebook, on claim and evidence pairs, three seeds, on the raw evidence and on the evidence with formatting artefacts removed.

Kaggle settings: **GPU T4**, **Internet on**. Run all cells (about one hour).

In [1]:
# Settings
REPO_URL = "https://github.com/MHKhan50/multilingual-factcheck-artefacts.git"
REPO_REF = "main"
SEEDS = "0,1,2"
VARIANTS = "raw,cleaned"
MODEL = "intfloat/multilingual-e5-base"
OUT = "/kaggle/working/ru22fact-neural-me5"

In [2]:
import subprocess, sys, torch
print("torch", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
import transformers; print("transformers", transformers.__version__)

torch 2.11.0+cu130 CUDA: True
GPU: Tesla T4
transformers 5.20.2


In [3]:
# Code and data (RU22Fact is downloaded from its pinned commit and checked by SHA-256)
import os
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "repo"], check=True)
subprocess.run(["git", "clone", "-q", "--branch", REPO_REF, REPO_URL, "repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-download", "--data-dir", "data/raw/ru22fact"], check=True)

8a6fb4de881c3702c0429a9d503da8d09e591be3

{
  "data_dir": "data/raw/ru22fact",
  "sha256": {
    "train": "9dc0f12d5d3b1a4a88a437c5630f216a31196ffd32b3ffac5d815c41d4726db8",
    "validate": "9b6e6b38b478204a28229791fde50385854b12309f2d60a49159c7f9f159f658",
    "test": "946b548f511deefac6898aad02189c809653e017aeda0f1f439def830a8a3aaf"
  }
}


CompletedProcess(args=['/usr/bin/python3', '-m', 'civintel_agentbench.cli', 'ru22fact-download', '--data-dir', 'data/raw/ru22fact'], returncode=0)

In [4]:
# Smoke run (about a minute): catches errors before the long run
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-neural", "--model", MODEL, "--data-dir", "data/raw/ru22fact",
                "--output", "/kaggle/working/smoke", "--seeds", "0", "--variants", "raw", "--epochs", "1", "--limit", "20"], check=True)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2556.92it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "raw", "seed": 0, "best_epoch": 1, "minutes": 0.0, "test_macro_f1": 0.2738}
{
  "raw": {
    "seeds": [
      0
    ],
    "macro_f1": {
      "mean": 0.27382113821138215,
      "std": 0.0
    },
    "per_language": {
      "en": {
        "mean": 0.2683982683982684,
        "std": 0.0
      },
      "ru": {
        "mean": 0.3333333333333333,
        "std": 0.0
      },
      "uk": {
        "mean": 0.23076923076923078,
        "std": 0.0
      },
      "zh": {
        "mean": 0.20512820512820515,
        "std": 0.0
      }
    },
    "minutes": 0.0
  }
}


CompletedProcess(args=['/usr/bin/python3', '-m', 'civintel_agentbench.cli', 'ru22fact-neural', '--model', 'intfloat/multilingual-e5-base', '--data-dir', 'data/raw/ru22fact', '--output', '/kaggle/working/smoke', '--seeds', '0', '--variants', 'raw', '--epochs', '1', '--limit', '20'], returncode=0)

In [5]:
# Full run: 2 variants x 3 seeds
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-neural", "--model", MODEL, "--data-dir", "data/raw/ru22fact",
                "--output", OUT, "--seeds", SEEDS, "--variants", VARIANTS], check=True)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2768.23it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "raw", "seed": 0, "best_epoch": 3, "minutes": 9.4, "test_macro_f1": 0.695}


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2706.83it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "raw", "seed": 1, "best_epoch": 2, "minutes": 9.2, "test_macro_f1": 0.7103}


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2431.68it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "raw", "seed": 2, "best_epoch": 3, "minutes": 9.4, "test_macro_f1": 0.6986}


... (3 outputs left out)

{"variant": "cleaned", "seed": 1, "best_epoch": 2, "minutes": 9.2, "test_macro_f1": 0.7024}


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2555.98it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "cleaned", "seed": 2, "best_epoch": 3, "minutes": 9.4, "test_macro_f1": 0.688}
{
  "cleaned": {
    "seeds": [
      0,
      1,
      2
    ],
    "macro_f1": {
      "mean": 0.6966278509643532,
      "std": 0.00763571303492057
    },
    "per_language": {
      "en": {
        "mean": 0.6471325021330196,
        "std": 0.009337884941154053
      },
      "ru": {
        "mean": 0.9971677730819138,
        "std": 0.002472460295766788
      },
      "uk": {
        "mean": 0.9990588855210785,
        "std": 0.0008545695563896295
      },
      "zh": {
        "mean": 0.7103820280227185,
        "std": 0.01489977731121942
      }
    },
    "minutes": 9.333333333333334
  },
  "raw": {
    "seeds": [
      0,
      1,
      2
    ],
    "macro_f1": {
      "mean": 0.7012770547818167,
      "std": 0.007969577580367359
    },
    "per_language": {
      "en": {
        "mean": 0.6519824255567416,
        "std": 0.009475792565696936
      },
      "ru": {
        "mean": 0.99868

CompletedProcess(args=['/usr/bin/python3', '-m', 'civintel_agentbench.cli', 'ru22fact-neural', '--model', 'intfloat/multilingual-e5-base', '--data-dir', 'data/raw/ru22fact', '--output', '/kaggle/working/ru22fact-neural-me5', '--seeds', '0,1,2', '--variants', 'raw,cleaned'], returncode=0)

In [6]:
import json
summary = json.load(open(OUT + "/summary.json"))
for variant, s in summary.items():
    langs = ", ".join("{} {:.3f}".format(k, v["mean"]) for k, v in s["per_language"].items())
    print("{:8s} macro-F1 {:.3f} +/- {:.3f} | {} | {:.0f} min per seed".format(
        variant, s["macro_f1"]["mean"], s["macro_f1"]["std"], langs, s["minutes"]))

cleaned  macro-F1 0.697 +/- 0.008 | en 0.647, ru 0.997, uk 0.999, zh 0.710 | 9 min per seed
raw      macro-F1 0.701 +/- 0.008 | en 0.652, ru 0.999, uk 0.999, zh 0.709 | 9 min per seed
